In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

try:
    import sweetviz as sv
except ImportError:
    pass

In [ ]:
# load the data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

In [ ]:
# prep data for modeling by separating X and y and dropping ids
test_ids = test['id']

train = train.drop(columns=['id'])
test = test.drop(columns=['id'])

X = train.drop(columns=['Irrigation_Need'])
y = train['Irrigation_Need']

# encode categorical target into integers
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# one hot encode the categorical variables across both sets to align shapes
X_all = pd.concat([X, test], axis=0)
X_all_encoded = pd.get_dummies(X_all, drop_first=True)

X_encoded = X_all_encoded.iloc[:len(train)]
test_encoded = X_all_encoded.iloc[len(train):]

# basic train test split using stratification to protect distribution
X_train, X_val, y_train, y_val = train_test_split(
    X_encoded, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# scale continuous variables
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
test_scaled = scaler.transform(test_encoded)

# calculate weights because target has huge class imbalance
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

In [ ]:
# bagging model based on random forest
rf = RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1)
rf.fit(X_train_scaled, y_train)

rf_preds = rf.predict(X_val_scaled)
print(f'RF Balanced Accuracy: {balanced_accuracy_score(y_val, rf_preds):.4f}')
print(classification_report(y_val, rf_preds, target_names=le.classes_))

In [ ]:
# boosting model based on xgboost
xgb = XGBClassifier(random_state=42, eval_metric='mlogloss', n_jobs=-1)
xgb.fit(X_train_scaled, y_train, sample_weight=sample_weights)

xgb_preds = xgb.predict(X_val_scaled)
print(f'XGB Balanced Accuracy: {balanced_accuracy_score(y_val, xgb_preds):.4f}')
print(classification_report(y_val, xgb_preds, target_names=le.classes_))

In [ ]:
# try a quick grid search just to optimize models a little bit more
rf_params = {"n_estimators": [100], "max_depth": [10, 15]}
rf_grid = GridSearchCV(RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1), rf_params, cv=3, scoring="balanced_accuracy")
rf_grid.fit(X_train_scaled, y_train)

xgb_params = {"learning_rate": [0.05, 0.1], "max_depth": [3, 5]}
xgb_grid = GridSearchCV(XGBClassifier(random_state=42, eval_metric="mlogloss", n_jobs=-1), xgb_params, cv=3, scoring="balanced_accuracy")
xgb_grid.fit(X_train_scaled, y_train) 

print("Best RF params:", rf_grid.best_params_)
print("Best XGB params:", xgb_grid.best_params_)

In [ ]:
# fit the best models to the full dataset for kaggle submission
X_full_scaled = scaler.fit_transform(X_encoded)

best_rf = rf_grid.best_estimator_
best_rf.fit(X_full_scaled, y_encoded)
test_preds_rf = best_rf.predict(test_scaled)

sub_rf = pd.DataFrame({'id': test_ids, 'Irrigation_Need': le.inverse_transform(test_preds_rf)})
sub_rf.to_csv('submission_rf.csv', index=False)

best_xgb = xgb_grid.best_estimator_
full_weights = compute_sample_weight(class_weight='balanced', y=y_encoded)
best_xgb.fit(X_full_scaled, y_encoded, sample_weight=full_weights)
test_preds_xgb = best_xgb.predict(test_scaled)

sub_xgb = pd.DataFrame({'id': test_ids, 'Irrigation_Need': le.inverse_transform(test_preds_xgb)})
sub_xgb.to_csv('submission_xgb.csv', index=False)